# Notebook 3B — AWS Glue PySpark ETL

**Purpose:** show how the same notebook logic from Use Case 2 becomes a repeatable **AWS Glue PySpark** pipeline.

## Where this notebook should be run
- **AWS Glue Studio notebook**, or
- **AWS Glue interactive session**


## Dependencies and AWS context

### Glue / Spark dependencies
- **SparkSession** — entry point for distributed processing
- **pyspark.sql.functions** — used for date parsing, null handling, derived columns, and aggregations

### One-time setup before the live Glue demo
Update the **bucket** and optional **prefix** in the config cell below.
Once updated, the notebook points to the exact S3 locations created by Use Case 1 and Use Case 2.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

# -------------------------------
# Config
# -------------------------------
AWS_REGION = os.getenv('AWS_REGION', 'ap-south-1')

# Direct S3 paths (no placeholders, no prefix confusion)
INPUT_PATH = "s3://usecase-etl-1/processed/retail_exploration_ready.csv"

OUT_DAILY = "s3://usecase-etl-2/daily_country_revenue_glue/"
OUT_MONTHLY = "s3://usecase-etl-2/monthly_category_revenue_glue/"

# -------------------------------
# Spark Session
# -------------------------------
spark = SparkSession.builder.appName('Glue-UseCase2-ETL').getOrCreate()

print('Glue input path  :', INPUT_PATH)
print('Glue daily output:', OUT_DAILY)
print('Glue monthly output:', OUT_MONTHLY)

Glue input path  : s3://usecase-etl-1/processed/retail_exploration_ready.csv
Glue daily output: s3://usecase-etl-2/daily_country_revenue_glue/
Glue monthly output: s3://usecase-etl-2/monthly_category_revenue_glue/


## Step 1 — Extract from S3

Glue reads the prepared file from a shared S3 location so every worker can access the same input.


In [4]:
raw_df = (
    spark.read
         .option('header', 'true')
         .option('inferSchema', 'true')
         .csv(INPUT_PATH)
)

print('Raw row count:', raw_df.count())
raw_df.show(5, truncate=False)


Raw row count: 500
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889.0   |Belgium       |2011-02-01 11:08:00|
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943.0   |Germany       |2011-01-28 11:32:00|
|536366   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|6       |02/05/2011 08:48|5.8      |18065.0   |Netherlands   |2011-02-05 08:48:00|
|536366   |22752    |SET 7 BABUSHKA NESTING BOXES     |4       |01/13/2011 13:54|7.55     |14512.0   |United Kingdom|2011-01-13 13:54:00|
|536366   |2173

## Step 2 — Transform and clean with PySpark

This cell mirrors the logic from Notebook 2 and Notebook 3, but in a distributed engine.


In [5]:
clean_df = (
    raw_df
      .withColumn('InvoiceDateTs', F.to_timestamp(F.col('InvoiceDate'), 'MM/dd/yyyy HH:mm'))
      .withColumn('Description', F.coalesce(F.col('Description'), F.lit('UNKNOWN_ITEM')))
      .withColumn('CustomerID', F.coalesce(F.col('CustomerID').cast('string'), F.lit('UNKNOWN_CUSTOMER')))
      .filter(F.col('InvoiceDateTs').isNotNull())
      .filter(F.col('UnitPrice') > 0)
      .withColumn('Revenue', F.col('Quantity') * F.col('UnitPrice'))
      .withColumn('IsReturn', F.col('Quantity') < 0)
      .withColumn('TransactionDate', F.to_date('InvoiceDateTs'))
      .withColumn('Month', F.date_format('InvoiceDateTs', 'yyyy-MM'))
      .withColumn('Category', F.split(F.col('Description'), ' ').getItem(0))
)

clean_df.show(5, truncate=False)


+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+-------------------+-------+--------+---------------+-------+--------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |InvoiceDateParsed  |InvoiceDateTs      |Revenue|IsReturn|TransactionDate|Month  |Category|
+---------+---------+---------------------------------+--------+----------------+---------+----------+--------------+-------------------+-------------------+-------+--------+---------------+-------+--------+
|536365   |71053    |WHITE METAL LANTERN              |6       |02/01/2011 11:08|5.49     |17889.0   |Belgium       |2011-02-01 11:08:00|2011-02-01 11:08:00|32.94  |false   |2011-02-01     |2011-02|WHITE   |
|536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER|2       |01/28/2011 11:32|4.22     |16943.0   |Germany       |2011-01-28 11:32:00|2011-01-28 11:32:00|8.44   |fal

## Step 3 — Aggregate into ETL outputs

This is the distributed equivalent of the pandas `groupby` logic.


In [6]:
daily_country_revenue = (
    clean_df.groupBy('TransactionDate', 'Country')
            .agg(F.round(F.sum('Revenue'), 2).alias('Revenue'))
)

monthly_category_revenue = (
    clean_df.groupBy('Month', 'Category')
            .agg(F.round(F.sum('Revenue'), 2).alias('Revenue'))
)

daily_country_revenue.show(10, truncate=False)
monthly_category_revenue.show(10, truncate=False)


+---------------+-----------+-------+
|TransactionDate|Country    |Revenue|
+---------------+-----------+-------+
|2011-02-05     |Netherlands|46.8   |
|2011-01-30     |France     |93.55  |
|2011-02-02     |Belgium    |121.02 |
|2011-03-14     |Spain      |6.76   |
|2011-01-13     |Germany    |72.92  |
|2011-01-25     |Spain      |31.32  |
|2011-03-12     |France     |20.88  |
|2011-03-28     |Spain      |54.7   |
|2011-02-13     |Spain      |18.39  |
|2011-03-26     |Germany    |82.34  |
+---------------+-----------+-------+
only showing top 10 rows

+-------+--------+-------+
|Month  |Category|Revenue|
+-------+--------+-------+
|2011-02|GLASS   |619.36 |
|2011-01|WHITE   |688.86 |
|2011-03|GLASS   |765.68 |
|2011-03|RED     |525.5  |
|2011-03|KNITTED |774.77 |
|2011-03|HAND    |1132.78|
|2011-02|CREAM   |408.47 |
|2011-02|SET     |552.78 |
|2011-01|CREAM   |314.82 |
|2011-01|SET     |538.56 |
+-------+--------+-------+
only showing top 10 rows


## Step 4 — Write outputs back to S3

Glue writes folders to S3 because Spark outputs distributed files rather than a single local CSV.


In [7]:
daily_country_revenue.write.mode('overwrite').option('header', 'true').csv(OUT_DAILY)
monthly_category_revenue.write.mode('overwrite').option('header', 'true').csv(OUT_MONTHLY)

print('Glue ETL outputs written to:')
print(OUT_DAILY)
print(OUT_MONTHLY)


Glue ETL outputs written to:
s3://usecase-etl-2/daily_country_revenue_glue/
s3://usecase-etl-2/monthly_category_revenue_glue/
